In [1]:
%load_ext autoreload
%autoreload 2

# Test Localization Pipeline

This notebook tests the full localization pipeline:
- **SequencePlaceRecognitionPipeline** for image-based place recognition
- **RansacPointCloudRegistrationPipeline** for point cloud registration
- **LocalizationPipeline** combining both

We run inference on the first 100 queries from map2 and save sample outputs.


## 1. Setup and Imports


In [14]:
from __future__ import annotations

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import Tensor
from torchvision import transforms as T
from PIL import Image
from tqdm import tqdm

# Import from mmpr.inference (our new module)
from mmpr.inference import (
    FaissFlatIndex,
    SequencePlaceRecognitionPipeline,
    RansacPointCloudRegistrationPipeline,
    LocalizationPipeline,
    LocalizationResult,
)

# place recognition model
from mmpr.models.megaloc import MegaLoc

# For loading query point clouds
from mmpr.data.pcd import SimplePCDLoader
from mmpr.data.transforms import get_T_map_to_world


In [15]:
# Configuration
ROOT_DATA_DIR = Path(
    "/mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps"  # ! CHANGE THIS PATH
)

# Database (map1) and Query (map2) directories
DB_MAP_DIR = ROOT_DATA_DIR / "map1" / "keyframe_map" / "keyframe_map"
QUERY_MAP_DIR = ROOT_DATA_DIR / "map3" / "keyframe_map" / "keyframe_map"

# Output directory for results
OUTPUT_DIR = Path("../experiments/localization_test").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline parameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_QUERIES = 100   # Test on first X queries
MAX_WINDOW = 25    # Sequence window size
PER_FRAME_K = 10   # Top-k per frame for sequence PR
FINAL_K = 5       # Final top-k candidates for localization
VOXEL_SIZE = 0.3   # Registration voxel downsample size

# Validate paths
assert ROOT_DATA_DIR.exists(), f"Path {ROOT_DATA_DIR} does not exist"
assert DB_MAP_DIR.exists(), f"Path {DB_MAP_DIR} does not exist"
assert QUERY_MAP_DIR.exists(), f"Path {QUERY_MAP_DIR} does not exist"

print(f"Device: {DEVICE}")
print(f"DB Map: {DB_MAP_DIR}")
print(f"Query Map: {QUERY_MAP_DIR}")
print(f"Output Dir: {OUTPUT_DIR}")


Device: cuda
DB Map: /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/map1/keyframe_map/keyframe_map
Query Map: /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/map3/keyframe_map/keyframe_map
Output Dir: /home/kartashov_ga/projects/mmpr/multimodal-place-recognition/experiments/localization_test


## 2. Load Model and Index


In [16]:
# Load the model
pr_model = MegaLoc()
pr_model.eval()
pr_model.to(DEVICE)

total_params = sum(p.numel() for p in pr_model.parameters())
print(f"Model loaded. Parameters: {total_params:,}")


Using cache found in /home/kartashov_ga/.cache/torch/hub/gmberton_MegaLoc_main
Using cache found in /home/kartashov_ga/.cache/torch/hub/facebookresearch_dinov2_main


Model loaded. Parameters: 228,640,321


Note that the Index should be built with the same model that will be used for inference.
In this case, we load the previously built index.

In [17]:
# Load the FAISS index from the database
pr_index = FaissFlatIndex.load(DB_MAP_DIR)

print(f"Index loaded from {DB_MAP_DIR}")
print(f"  - Size: {pr_index.size()} entries")
print(f"  - Dimension: {pr_index.dim()}")
print(f"  - Metric: {pr_index.metric()}")


Index loaded from /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/map1/keyframe_map/keyframe_map
  - Size: 712 entries
  - Dimension: 8448
  - Metric: l2


## 3. Create Pipelines


In [18]:
# Create Sequence Place Recognition Pipeline
seq_pr_pipeline = SequencePlaceRecognitionPipeline(
    index=pr_index,
    model=pr_model,
    device=DEVICE,
    max_window=MAX_WINDOW,
    per_frame_k=PER_FRAME_K,
    final_k=FINAL_K,
)
print(f"SequencePlaceRecognitionPipeline created (window={MAX_WINDOW}, per_frame_k={PER_FRAME_K}, final_k={FINAL_K})")

# Create Registration Pipeline
reg_pipeline = RansacPointCloudRegistrationPipeline(
    voxel_downsample_size=VOXEL_SIZE,
)
print(f"RansacPointCloudRegistrationPipeline created (voxel_size={VOXEL_SIZE})")

# Create full Localization Pipeline
# Note: LocalizationPipeline uses PlaceRecognitionPipeline, but works with SequencePR too
loc_pipeline = LocalizationPipeline(
    index=pr_index,
    place_recognition=seq_pr_pipeline,  # Using single-frame PR for now
    registration=reg_pipeline,
    index_root=DB_MAP_DIR,
    require_db_pointcloud=False,  # Skip candidates without point clouds
)
print("LocalizationPipeline created")


SequencePlaceRecognitionPipeline created (window=25, per_frame_k=10, final_k=5)
RansacPointCloudRegistrationPipeline created (voxel_size=0.3)
LocalizationPipeline created


## 4. Load Query Data


In [19]:
# Image preprocessing for MegaLoc
image_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.Resize([322, 322], antialias=True),
])


def read_image(image_path: str | Path) -> Tensor:
    """Load and preprocess an image."""
    image = Image.open(image_path)
    image = image_transform(image)
    return image


def to_batch(image: Tensor) -> dict[str, Tensor]:
    """Convert image tensor to batch dict for model input."""
    return {"images_0": image.unsqueeze(0)}


In [20]:
# Load query point clouds using SimplePCDLoader
T_map_to_world = get_T_map_to_world("map2")
query_pcd_loader = SimplePCDLoader(
    map_root=QUERY_MAP_DIR,
    scans_subdir="scans",
    T_map_to_world=T_map_to_world,
)
print(f"Query PCD loader: {len(query_pcd_loader)} scans available")

# Get image directory
query_images_dir = QUERY_MAP_DIR / "zedxone_left" / "rgb"
if not query_images_dir.exists():
    # Try alternative paths
    for alt in ["images", "rgb"]:
        alt_dir = QUERY_MAP_DIR / alt
        if alt_dir.exists():
            query_images_dir = alt_dir
            break

print(f"Query images directory: {query_images_dir}")
assert query_images_dir.exists(), f"Images directory not found: {query_images_dir}"

# Limit to NUM_QUERIES
num_available = min(len(query_pcd_loader), NUM_QUERIES)
print(f"Will process {num_available} queries")


Query PCD loader: 629 scans available
Query images directory: /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/map3/keyframe_map/keyframe_map/zedxone_left/rgb
Will process 100 queries


## 5. Run Inference


In [21]:
# Run localization on first NUM_QUERIES frames
results: list[LocalizationResult] = []
timing_stats: list[dict] = []
errors: list[dict] = []

print(f"Running localization on {num_available} queries...")
print(f"Using: LocalizationPipeline with PlaceRecognitionPipeline + RANSAC Registration")

qis = np.linspace(0, len(query_pcd_loader), num_available, dtype=int)
for qi in tqdm(qis, desc="Localizing"):
    # try:
    
    # Load query image
    image_path = query_images_dir / f"{qi:06d}.jpg"

    if not image_path.exists():
        # Try png
        image_path = query_images_dir / f"{qi:06d}.png"
    
    if not image_path.exists():
        errors.append({"query_idx": qi, "error": f"Image not found: {image_path}"})
        continue
    
    image = read_image(image_path)
    pr_input = to_batch(image)
    pr_input = {k: v.to(DEVICE) for k, v in pr_input.items()}

    # Load query point cloud
    query_points, query_pose7, query_pcd_path = query_pcd_loader[qi]

    query_pc = torch.from_numpy(query_points).float()
    
    # Run localization
    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t_start = time.perf_counter()
    
    result = loc_pipeline.infer(
        pr_input=pr_input,
        query_pc=query_pc,
        k=FINAL_K,
    )

    torch.cuda.synchronize() if DEVICE == "cuda" else None
    t_end = time.perf_counter()
    
    results.append(result)
    timing_stats.append({
        "query_idx": qi,
        "time_ms": (t_end - t_start) * 1000,
        "num_candidates": len(result.candidates),
        "chosen_idx": result.chosen_idx,
    })
        
    # except Exception as e:
    #     errors.append({"query_idx": qi, "error": str(e)})
    #     continue

print(f"\nCompleted: {len(results)} successful, {len(errors)} errors")


Running localization on 100 queries...
Using: LocalizationPipeline with PlaceRecognitionPipeline + RANSAC Registration


Localizing:  39%|███▉      | 39/100 [00:44<00:54,  1.13it/s]

[Open3D WARNING] Too few correspondences (99) after mutual filter, fall back to original correspondences.


Localizing:  40%|████      | 40/100 [00:45<00:52,  1.14it/s]

[Open3D WARNING] Too few correspondences (111) after mutual filter, fall back to original correspondences.


...

[Open3D WARNING] Too few correspondences (138) after mutual filter, fall back to original correspondences.


Localizing:  96%|█████████▌| 96/100 [01:41<00:04,  1.15s/it]

[Open3D WARNING] Too few correspondences (154) after mutual filter, fall back to original correspondences.


Localizing: 100%|██████████| 100/100 [01:45<00:00,  1.05s/it]


Completed: 99 successful, 1 errors


In [22]:
errors

[{'query_idx': np.int64(629),
  'error': 'Image not found: /mnt/external_usb_hdd/6YL/Datasets/SberRobotics/maps/map3/keyframe_map/keyframe_map/zedxone_left/rgb/000629.png'}]

In [23]:
# Timing statistics
if timing_stats:
    times = [s["time_ms"] for s in timing_stats]
    print(f"Timing Statistics (ms):")
    print(f"  Mean: {np.mean(times):.2f}")
    print(f"  Std:  {np.std(times):.2f}")
    print(f"  Min:  {np.min(times):.2f}")
    print(f"  Max:  {np.max(times):.2f}")
    print(f"  Median: {np.median(times):.2f}")


Timing Statistics (ms):
  Mean: 988.85
  Std:  230.35
  Min:  572.60
  Max:  1577.83
  Median: 964.64


## 6. Save Results


In [24]:
# Create output subdirectory for this run
run_dir = OUTPUT_DIR / f"run_{time.strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)
results_dir = run_dir / "localization_results"
results_dir.mkdir(exist_ok=True)

# Save individual LocalizationResults as JSON
print(f"Saving {len(results)} localization results to {results_dir}")
for i, result in enumerate(results):
    result_path = results_dir / f"result_{qis[i]:04d}.json"
    result.save(result_path)

# Save timing statistics
timing_df = pd.DataFrame(timing_stats)
timing_path = run_dir / "timing_stats.csv"
timing_df.to_csv(timing_path, index=False)
print(f"Timing stats saved to {timing_path}")

# Save errors
if errors:
    errors_path = run_dir / "errors.json"
    errors_path.write_text(json.dumps(errors, indent=2))
    print(f"Errors saved to {errors_path}")

# Save summary
summary = {
    "num_queries": num_available,
    "num_successful": len(results),
    "num_errors": len(errors),
    "config": {
        "device": DEVICE,
        "max_window": MAX_WINDOW,
        "per_frame_k": PER_FRAME_K,
        "final_k": FINAL_K,
        "voxel_size": VOXEL_SIZE,
    },
    "timing_ms": {
        "mean": float(np.mean(times)) if timing_stats else None,
        "std": float(np.std(times)) if timing_stats else None,
        "min": float(np.min(times)) if timing_stats else None,
        "max": float(np.max(times)) if timing_stats else None,
        "median": float(np.median(times)) if timing_stats else None,
    },
    "db_map": str(DB_MAP_DIR),
    "query_map": str(QUERY_MAP_DIR),
}
summary_path = run_dir / "summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"Summary saved to {summary_path}")

print(f"\n=== All outputs saved to {run_dir} ===")


Saving 99 localization results to /home/kartashov_ga/projects/mmpr/multimodal-place-recognition/experiments/localization_test/run_20251217_083742/localization_results
Timing stats saved to /home/kartashov_ga/projects/mmpr/multimodal-place-recognition/experiments/localization_test/run_20251217_083742/timing_stats.csv


TypeError: Object of type int64 is not JSON serializable

In [25]:
# Verify we can load the saved results back
if results:
    sample_path = results_dir / "result_0000.json"
    loaded_result = LocalizationResult.load(sample_path)
    
    print("Sample loaded result:")
    print(f"  Version: {loaded_result.version}")
    print(f"  Chosen idx: {loaded_result.chosen_idx}")
    print(f"  Num candidates: {len(loaded_result.candidates)}")
    
    if loaded_result.candidates:
        c = loaded_result.candidates[0]
        print(f"  First candidate:")
        print(f"    - idx: {c.idx}")
        print(f"    - pr_distance: {c.pr_distance:.4f}")
        print(f"    - db_pose: {c.db_pose}")
        print(f"    - estimated_pose: {c.estimated_pose}")
        print(f"    - registration_confidence: {c.registration_confidence}")


Sample loaded result:
  Version: 1
  Chosen idx: 0
  Num candidates: 5
  First candidate:
    - idx: 0
    - pr_distance: 0.7930
    - db_pose: [ 0.0440825  -0.0384556   0.01084682 -0.00194747 -0.01912343 -0.3780022
  0.9256051 ]
    - estimated_pose: [ 0.0455174  -0.08979867  0.02957725  0.00099169 -0.0173078  -0.33620899
  0.94162783]
    - registration_confidence: 1.0


# ⚠️ WARNING: This notebook is not complete. It is a work in progress.

It is working fine, but we need to:
- add more documentation
- add tutorial on how to build the index
- switch to the fish-eye camera images
